# Distlib Package Examples

This notebook shows five examples of the `distlib` package.

Distlib provides utilities for Python package distribution.


## Example 1: Package version parsing and comparison

In [ ]:
# Example: Version handling with distlib
from distlib.version import NormalizedVersion, UnsupportedVersionError

# Test version strings
version_strings = [
    "1.0",
    "1.0.0", 
    "1.0.0a1",
    "1.0.0.dev0",
    "2.1.3",
    "1.0.0b2",
    "1.0.0rc1",
    "1.0.0.post1",
    "1.0.0+local.1",
    "invalid.version",
    "1.0.0.0.0",  # Too many parts
    ""
]

print("Testing distlib NormalizedVersion:")
print("-" * 50)

for vs in version_strings:
    try:
        nv = NormalizedVersion(vs)
        print(f"✓ '{vs}' -> {nv}")
    except UnsupportedVersionError as e:
        print(f"✗ '{vs}' -> UnsupportedVersionError: {e}")
    except Exception as e:
        print(f"✗ '{vs}' -> Error: {type(e).__name__}: {e}")
        raise

print("\n" + "=" * 50)
print("Version comparison examples:")
print("-" * 50)

# Test version comparisons
valid_versions = []
rejected = []
for vs in version_strings:
    try:
        valid_versions.append((vs, NormalizedVersion(vs)))
    except UnsupportedVersionError:
        rejected.append(vs)

# Exactly the two deliberately malformed strings are rejected (a PEP 440 release may have any number of parts).
assert rejected == ["invalid.version", ""], rejected

# Compare some versions
if len(valid_versions) >= 2:
    v1_str, v1 = valid_versions[0]
    v2_str, v2 = valid_versions[1]
    
    print(f"Comparing '{v1_str}' and '{v2_str}':")
    print(f"  {v1} < {v2}: {v1 < v2}")
    print(f"  {v1} == {v2}: {v1 == v2}")
    print(f"  {v1} > {v2}: {v1 > v2}")

# Test sorting versions
print(f"\nSorting valid versions:")
sorted_versions = sorted(valid_versions, key=lambda x: x[1])
for orig, norm in sorted_versions:
    print(f"  {orig} -> {norm}")

## Example 2: Working with wheel files and metadata

In [ ]:
from distlib.wheel import Wheel
from distlib.metadata import Metadata
import tempfile
import os
import io

# Create sample metadata using the correct distlib API
# Method 1: Create metadata from a string
metadata_content = """\
Name: example-package
Version: 1.0.0
Summary: An example package for testing
Author: Test Author
Author-email: test@example.com
License: MIT
Home-page: https://github.com/example/example_package
Description: A longer description of the example package
"""

print("Creating metadata from string content...")
try:
    # Create metadata from string (distlib reads it at construction: Metadata(fileobj=...))
    metadata = Metadata(fileobj=io.StringIO(metadata_content))
    
    print("✓ Metadata created successfully from string")
    
    print("\nPackage metadata:")
    print(f"Name: {metadata.name}")
    print(f"Version: {metadata.version}")
    print(f"Summary: {metadata.summary}")
    
    assert (metadata.name, metadata.version) == ("example-package", "1.0.0")
    assert metadata.summary == "An example package for testing"
    
    # The other PKG-INFO fields are not attributes of Metadata; todict() carries every parsed field
    fields = metadata.todict()
    print(f"Author: {fields['author']}")
    print(f"Author Email: {fields['author_email']}")
    print(f"License: {fields['license']}")
    print(f"Home Page: {fields['home_page']}")
    assert fields['author'] == "Test Author" and fields['license'] == "MIT"
        
except Exception as e:
    print(f"✗ Failed to create metadata: {e}")
    raise

# Test metadata serialization
print("\nMetadata serialization:")
try:
    # Convert to string format
    metadata_str = str(metadata)
    print(f"Metadata as string (first 200 chars): {metadata_str[:200]}...")
    print(f"String length: {len(metadata_str)}")
    
    # Test writing to file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        temp_path = f.name
        try:
            metadata.write(fileobj=f, legacy=True)  # legacy=True: PKG-INFO key/value format, not JSON
            f.flush()
            with open(temp_path) as written:
                written_text = written.read()
            assert "Name: example-package" in written_text and "Version: 1.0.0" in written_text, written_text
            print(f"✓ Successfully wrote metadata to {temp_path}")
        except Exception as write_error:
            print(f"✗ Write error: {write_error}")
            raise
        finally:
            # Clean up
            if os.path.exists(temp_path):
                os.unlink(temp_path)
    
except Exception as e:
    print(f"Serialization error: {e}")
    raise

# Alternative method: Create metadata from dictionary
print("\nAlternative: Creating metadata from mapping...")
try:
    metadata_dict = {
        'name': 'example-package',
        'version': '1.0.0',
        'summary': 'An example package for testing',
        'author': 'Test Author',
        'author_email': 'test@example.com',
        'license': 'MIT'
    }
    
    # Try creating from mapping if supported
    if hasattr(Metadata, 'from_dict'):
        metadata2 = Metadata.from_dict(metadata_dict)
        print("✓ Created metadata from dictionary")
    else:
        # Manual approach: create PKG-INFO format string
        pkg_info_lines = []
        field_mapping = {
            'name': 'Name',
            'version': 'Version', 
            'summary': 'Summary',
            'author': 'Author',
            'author_email': 'Author-email',
            'license': 'License'
        }
        
        for key, value in metadata_dict.items():
            if key in field_mapping:
                pkg_info_lines.append(f"{field_mapping[key]}: {value}")
        
        pkg_info_content = '\n'.join(pkg_info_lines)
        print(f"Generated PKG-INFO content:\n{pkg_info_content}")
        
        metadata2 = Metadata(fileobj=io.StringIO(pkg_info_content))
        assert metadata2.todict()['author_email'] == metadata_dict['author_email']
        print("✓ Created metadata from generated PKG-INFO")
        
except Exception as e:
    print(f"✗ Alternative method failed: {e}")
    raise

# Test wheel filename parsing
print("\nWheel filename parsing:")
wheel_filenames = [
    "example_package-1.0.0-py3-none-any.whl",
    "numpy-1.21.0-cp39-cp39-win_amd64.whl",
    "requests-2.25.1-py2.py3-none-any.whl"
]

# (name, version, python tags, ABI tags, platform tags) each filename encodes -- measured, distlib 0.4.3
expected_parses = {
    "example_package-1.0.0-py3-none-any.whl": ("example_package", "1.0.0", ["py3"], ["none"], ["any"]),
    "numpy-1.21.0-cp39-cp39-win_amd64.whl": ("numpy", "1.21.0", ["cp39"], ["cp39"], ["win_amd64"]),
    "requests-2.25.1-py2.py3-none-any.whl": ("requests", "2.25.1", ["py2", "py3"], ["none"], ["any"]),
}

for filename in wheel_filenames:
    # distlib parses a wheel filename when a Wheel is built from it (there is no Wheel.parse_filename).
    wheel = Wheel(filename)
    parsed = (wheel.name, wheel.version, wheel.pyver, wheel.abi, wheel.arch)
    assert parsed == expected_parses[filename], (filename, parsed)
    print(f"✓ Parsed '{filename}': name={wheel.name}, version={wheel.version}, "
          f"tags={'.'.join(wheel.pyver)}-{'.'.join(wheel.abi)}-{'.'.join(wheel.arch)}")

# Test creating a wheel object
print("\nWheel object creation:")
try:
    with tempfile.TemporaryDirectory() as tmpdir:
        # Create a dummy wheel file path
        wheel_path = os.path.join(tmpdir, "example_package-1.0.0-py3-none-any.whl")
        
        # Try to create wheel object
        if hasattr(Wheel, '__init__'):
            print(f"Wheel class is available")
            print(f"Would work with wheel at: {wheel_path}")
            
            # Show wheel-related functionality
            wheel_methods = [m for m in dir(Wheel) if not m.startswith('_') and callable(getattr(Wheel, m, None))]
            print(f"Available Wheel methods: {wheel_methods[:10]}...")  # Show first 10
        
except Exception as e:
    print(f"Wheel object creation error: {e}")
    raise

# Final metadata inspection
print("\nFinal metadata inspection:")
try:
    if 'metadata' in locals():
        print(f"Metadata object exists: {type(metadata)}")
        
        # Try to access common attributes
        common_attrs = ['name', 'version', 'summary', 'description']
        for attr in common_attrs:
            try:
                value = getattr(metadata, attr, None)
                print(f"  {attr}: {value}")
            except Exception as attr_error:
                print(f"  {attr}: Error accessing - {attr_error}")
                raise
                
        # Show all populated fields (probing every dir() attribute raises: `dependencies` is not
        # implemented for legacy PKG-INFO metadata)
        attrs = sorted(metadata.todict())
        if attrs:
            print(f"Available attributes: {attrs}")
        assert {'name', 'version', 'author', 'home_page'} <= set(attrs), attrs
        
except Exception as e:
    print(f"Final inspection error: {e}")
    raise

print(f"\nDistlib test completed!")

## Example 3: ScriptMaker writes and runs a console-script launcher

In [ ]:
# Example 3: ScriptMaker writes a console-script launcher for an entry point (what installers do for
# every `console_scripts` entry), and the launcher runs under this interpreter.
import os
import subprocess
import sys
import tempfile
from distlib.scripts import ScriptMaker
from distlib.util import get_export_entry

entry = get_export_entry("demo = json.tool:main")
assert (entry.name, entry.prefix, entry.suffix, entry.flags) == ("demo", "json.tool", "main", []), entry
print(f"Entry point: name={entry.name} module={entry.prefix} function={entry.suffix}")

with tempfile.TemporaryDirectory() as tmpdir:
    maker = ScriptMaker(None, tmpdir)
    maker.variants = {""}  # only `demo`; the default also writes `demo-X.Y`
    files = maker.make("demo = json.tool:main")
    assert files == [os.path.join(tmpdir, "demo")], files
    assert os.access(files[0], os.X_OK), oct(os.stat(files[0]).st_mode)
    with open(files[0]) as handle:
        head = handle.read(4096)
    shebang = head.splitlines()[0]
    # distlib writes `#!<interpreter>`, or a /bin/sh wrapper that execs the interpreter when its path is too long
    # for a shebang line or contains spaces
    assert shebang == f"#!{sys.executable}" or (shebang == "#!/bin/sh" and sys.executable in head), (shebang, sys.executable)
    plain = {**os.environ, "PYTHON_COLORS": "0", "NO_COLOR": "1"}  # Python 3.14's json.tool adds colour under FORCE_COLOR
    run = subprocess.run([files[0]], input='{"a": 1}', capture_output=True, text=True, timeout=120, env=plain)
assert (run.returncode, run.stdout) == (0, '{\n    "a": 1\n}\n'), (run.returncode, run.stdout, run.stderr)
print(f"ScriptMaker wrote {os.path.basename(files[0])} ({shebang}); running it ran json.tool:main")

## Example 4: Package installation utilities

In [ ]:
from distlib.util import get_platform, is_string_sequence
from distlib.compat import string_types
import sys
import os

# Platform detection
print("Platform detection:")
try:
    platform = get_platform()
    print(f"Current platform: {platform}")
    print(f"Platform type: {type(platform).__name__}")
except Exception as e:
    print(f"Platform detection error: {e}")
    raise

# System information
print(f"\nSystem information:")
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Platform: {sys.platform}")
print(f"Architecture: {os.uname().machine if hasattr(os, 'uname') else 'Unknown'}")

# String sequence validation
print(f"\nString sequence validation:")
# (input, expected result or the exception distlib raises for it)
test_inputs = [
    ("single string", True),              # a str iterates as one-character strings
    (["list", "of", "strings"], True),
    (("tuple", "of", "strings"), True),
    (["mixed", 123, "types"], False),
    (123, TypeError),                     # not iterable
    (None, TypeError),                    # not iterable
    ([], AssertionError),                 # distlib asserts the sequence is non-empty
]

for inp, expected in test_inputs:
    try:
        is_seq = is_string_sequence(inp)
    except (TypeError, AssertionError) as e:
        if not (isinstance(expected, type) and isinstance(e, expected)):
            raise
        print(f"{str(inp)[:50]:50} -> raises {type(e).__name__} (expected)")
    else:
        assert is_seq is expected, (inp, is_seq, expected)
        print(f"{str(inp)[:50]:50} -> {is_seq}")

# Test compatibility utilities
print(f"\nCompatibility utilities:")
test_strings = ["test", b"bytes", 123, None]

for test_val in test_strings:
    try:
        is_string = isinstance(test_val, string_types)
        print(f"{str(test_val):15} is string type: {is_string}")
    except Exception as e:
        print(f"{str(test_val):15} -> Error: {e}")
        raise

# Environment variables relevant to packaging
print(f"\nPackaging-related environment variables:")
env_vars = [
    'PYTHONPATH',
    'PYTHON_HOME', 
    'PIP_INDEX_URL',
    'VIRTUAL_ENV',
    'CONDA_DEFAULT_ENV'
]

for var in env_vars:
    # only report whether each is set: an index URL can embed credentials
    print(f"  {var}: {'set' if var in os.environ else 'not set'}")

# Test path utilities
print(f"\nPath utilities:")
import distlib.util as util

# Test various path operations if available
available_functions = [attr for attr in dir(util) if not attr.startswith('_') and callable(getattr(util, attr))]
print(f"Available utility functions: {len(available_functions)}")
print(f"Sample functions: {available_functions[:10]}")
assert {'get_platform', 'is_string_sequence', 'parse_name_and_version'} <= set(available_functions), available_functions


# Verify distlib util functionality
print(f"\nDistlib util module working: {hasattr(get_platform, '__call__')}")
print(f"String sequence validator working: {hasattr(is_string_sequence, '__call__')}")

## Example 5: Database and index operations

In [ ]:
from distlib.index import PackageIndex
from distlib.database import make_dist
import json

# Test package index operations
print("Package index operations:")
try:
    # Create a package index (typically points to PyPI)
    index = PackageIndex()
    print(f"Package index created: {type(index).__name__}")
    print(f"Index URL: {getattr(index, 'url', 'Not available')}")
    
    # Test index properties
    print(f"Index attributes:")
    attributes = [attr for attr in dir(index) if not attr.startswith('_')]
    print(f"  Available attributes: {len(attributes)}")
    print(f"  Sample attributes: {attributes[:10]}")
    
except Exception as e:
    print(f"Package index error: {e}")
    raise

# Test distribution creation
print(f"\nDistribution operations:")
try:
    # Create a sample distribution metadata
    dist_info = {
        'name': 'test-package',
        'version': '1.0.0',
        'summary': 'A test package'
    }
    
    print(f"Sample distribution info: {dist_info}")
    
    # Test make_dist function if available
    print(f"make_dist function available: {callable(make_dist)}")
    
    # Test creating a distribution object
    try:
        dist = make_dist(dist_info['name'], dist_info['version'])
        if dist:
            print(f"Distribution created: {dist}")
            print(f"Distribution name: {getattr(dist, 'name', 'N/A')}")
            print(f"Distribution version: {getattr(dist, 'version', 'N/A')}")
        else:
            print("Distribution creation returned None")
    except Exception as e:
        print(f"Distribution creation error: {e}")
        raise
    
except Exception as e:
    print(f"Distribution operations error: {e}")
    raise

# Test JSON handling for metadata
print(f"\nMetadata serialization:")
metadata_examples = [
    {
        "name": "example-package",
        "version": "1.2.3",
        "dependencies": ["requests", "click"],
        "metadata_version": "2.1"
    },
    {
        "name": "another-package", 
        "version": "0.1.0",
        "author": "Test Author",
        "license": "MIT"
    }
]

for i, metadata in enumerate(metadata_examples, 1):
    try:
        # Serialize to JSON
        json_str = json.dumps(metadata, indent=2)
        print(f"\nMetadata {i} as JSON:")
        print(json_str)
        
        # Parse back from JSON
        parsed = json.loads(json_str)
        print(f"JSON round-trip successful: {parsed == metadata}")
        
    except Exception as e:
        print(f"Metadata {i} serialization error: {e}")
        raise

# Test distlib module availability
print(f"\nDistlib module verification:")
distlib_modules = [
    'distlib.version',
    'distlib.metadata', 
    'distlib.wheel',
    'distlib.scripts',
    'distlib.util',
    'distlib.index',
    'distlib.database'
]

available_modules = []
for module_name in distlib_modules:
    __import__(module_name)
    available_modules.append(module_name)
    print(f"✓ {module_name}")

print(f"\nDistlib modules available: {len(available_modules)}/{len(distlib_modules)}")
assert available_modules == distlib_modules, available_modules
print("\nDistlib testing completed!")